# ReelForge - de vrais plans video IA (100% gratuit)

Ce notebook genere de **vraies videos IA** (des scenes qui bougent, pas des images
fixes) sur le **GPU gratuit de Google Colab**. Aucune carte bancaire, aucun credit,
aucune inscription : le GPU est fourni par Google, le modele est libre (Apache 2.0).

## La boucle complete
1. Dans ReelForge, ecris ton sujet et clique **« Prompts pour Colab »**.
2. Copie le JSON affiche et colle-le dans la cellule 3 ci-dessous.
3. Ici : `Execution` -> `Modifier le type d'execution` -> **T4 GPU** (gratuit), puis
   execute les cellules dans l'ordre (`Ctrl+F9`).
4. Telecharge `clips_ia.zip` a la fin.
5. Dans ReelForge, depose le zip dans **« Clips IA generes »** : la voix, les
   sous-titres et le montage sont ajoutes automatiquement.

> Modele par defaut : **Wan 2.1 (1.3B)**, licence Apache 2.0 — gratuit et
> commercialisable. Chaque clip fait ~5 s en 480x832. Compte 2 a 4 min par clip
> sur un T4 gratuit.

> Honnetete : Colab gratuit ne garantit ni la disponibilite du T4 ni un quota
> illimite. Si le GPU est indisponible, reessaie plus tard ou utilise un autre
> compte Google. Le reste de ReelForge (voix, sous-titres, montage, personnages
> 3D) tourne sans GPU et sans limite.


In [ ]:
# 1) Verifier que le GPU est bien disponible
!nvidia-smi
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "AUCUN GPU - active le T4 dans Execution > Modifier le type d'execution")


In [ ]:
# 2) Installer les dependances (2-3 minutes)
!pip install -q --upgrade "diffusers>=0.32" transformers accelerate imageio imageio-ffmpeg ftfy


In [ ]:
# 3) Coller ici le JSON du site (bouton « Prompts pour Colab »)
#    Laisser {} pour ecrire tes prompts a la main dans PROMPTS_MANUELS.
import json

API_JSON = r"""{}"""

PROMPTS_MANUELS = [
    # "Cinematic vertical shot of a rainy city street at night, neon reflections, slow push-in",
]

DEFAUTS = [
    "Cinematic vertical shot of a rainy city street at night, neon reflections on wet asphalt, slow camera push-in",
    "Close-up of coffee being poured into a cup, warm morning light, shallow depth of field, vertical framing",
    "Aerial view of a foggy mountain range at sunrise, slow drone movement, vertical 9:16",
]

NEGATIF_DEFAUT = ("blurry, low quality, distorted, watermark, text, subtitles, caption, logo, "
                  "still image, slideshow")
REGLAGES_DEFAUT = {"model": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers", "height": 480,
                   "width": 832, "num_frames": 81, "fps": 16, "guidance": 5.0}

cfg = json.loads(API_JSON) if API_JSON.strip() not in ("", "{}") else {}
PROMPTS = PROMPTS_MANUELS or cfg.get("prompts") or DEFAUTS
NEGATIVE_PROMPT = cfg.get("negative_prompt") or NEGATIF_DEFAUT
S = {**REGLAGES_DEFAUT, **(cfg.get("settings") or {})}

MODEL_ID = S["model"]
HEIGHT, WIDTH = S["height"], S["width"]
NUM_FRAMES, FPS, GUIDANCE = S["num_frames"], S["fps"], S["guidance"]
CLIPS_PER_PROMPT = 1
SEED = 42

print(f"{len(PROMPTS)} prompt(s), modele {MODEL_ID}, {WIDTH}x{HEIGHT} @ {FPS} i/s")
for i, p in enumerate(PROMPTS):
    print(f"  [{i}] {p[:100]}")


In [ ]:
# 4) Charger le modele Wan 2.1 (1.3B, Apache 2.0, ~15 Go de telechargement)
import torch
from diffusers import AutoencoderKLWan, WanPipeline

pipe = WanPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)

# Le T4 (16 Go) est juste : on laisse diffusers placer les couches sur le GPU.
pipe.enable_model_cpu_offload()
print("Modele charge.")


In [ ]:
# 5) Generer les clips (2 a 4 min par clip sur un T4 gratuit)
from pathlib import Path
from diffusers.utils import export_to_video

OUT = Path("clips_ia"); OUT.mkdir(exist_ok=True)
gen = torch.Generator("cpu").manual_seed(SEED)

for p_index, prompt in enumerate(PROMPTS):
    for k in range(CLIPS_PER_PROMPT):
        print(f"Generation {p_index+1}/{len(PROMPTS)} : {prompt[:60]}...")
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            height=HEIGHT, width=WIDTH,
            num_frames=NUM_FRAMES, guidance_scale=GUIDANCE,
            generator=gen,
        ).frames[0]
        path = OUT / f"clip_{p_index:02d}_{k}.mp4"
        export_to_video(result, str(path), fps=FPS)
        print("  ->", path)

print("\nClips generes :")
!ls -lh clips_ia


In [ ]:
# 6) Telecharger l'archive a deposer dans ReelForge
import shutil
shutil.make_archive("clips_ia", "zip", "clips_ia")
from google.colab import files
files.download("clips_ia.zip")


## Alternatives gratuites

| Modele | Licence | VRAM | Remarque |
|---|---|---|---|
| **Wan 2.1 1.3B** (ce notebook) | Apache 2.0 | ~12 Go | Le plus accessible sur T4 |
| **LTX-Video 2B** | Apache 2.0 | ~12 Go | Beaucoup plus rapide (quasi temps reel) |
| **CogVideoX-2B** | Apache 2.0 | ~12 Go | Bonne qualite, plus lent |

Pour LTX-Video, remplace la cellule 4 par :

```python
from diffusers import LTXPipeline
pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
pipe.to("cuda")
```

> La generation video IA demande un GPU : c'est une contrainte materielle, pas une
> limite de ReelForge. Colab fournit ce GPU gratuitement ; ReelForge s'occupe du
> montage, de la voix, des sous-titres et des personnages 3D.
